In [70]:
from pathlib import Path

#
# MAIN
#

path = Path("./inputs")

In [71]:

from converter import doc2text

# convertie les documents en textes Markdown
doc2text(path)


Start scan : inputs
Fichier : budget_2024.csv (déjà traité)
Fichier : budget_2024.vec (déjà traité)
Fichier : Acceuil_affichage_Mairie de Triffouillis sur Loire.pdf (déjà traité)
Fichier : Acceuil_affichage_Mairie de Triffouillis sur Loire.vec (déjà traité)
Fichier : logo_triffouillis.vec (déjà traité)
Fichier : logo_triffouillis.webp (déjà traité)
Fichier : voeux2025_trifouillis.vec (déjà traité)
Fichier : voeux2025_trifouillis.wav (déjà traité)
Fichier : Demande d'information sur l'entretien de la voirie.docx (déjà traité)
Fichier : Demande d'information sur l'entretien de la voirie.vec (déjà traité)
Fichier : Demande de participation à la réunion municipale sur la revitalisation du centre-ville.docx (déjà traité)
Fichier : Demande de participation à la réunion municipale sur la revitalisation du centre-ville.vec (déjà traité)
Fichier : Signalement et demande d'intervention pour un éclairage public défectueux.docx (déjà traité)
Fichier : Signalement et demande d'intervention pour un 

In [72]:
#from embeddings import md2vec

from sentence_bert import make_embeddings as embeddings_SentenceBERT
from mistral import make_embeddings as embeddings_Mistral
from fast_text import make_embeddings as embeddings_FastText
from typing import Callable
from utils import split_markdown_by_headers
import numpy as np
import pickle
from pathlib import Path

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
  """Calcule la similarité cosinus entre deux vecteurs."""
  dot_product = np.dot(vec1, vec2)
  norm_vec1 = np.linalg.norm(vec1)
  norm_vec2 = np.linalg.norm(vec2)
  if norm_vec1 == 0 or norm_vec2 == 0:
    return 0 # Éviter la division par zéro
  return dot_product / (norm_vec1 * norm_vec2)


def save_embeddings(embedding_docs: list, embedding_name: str):
    """
    Sauvegarde l'embedding des documents dans un fichier pickle.
    """
    with open(embedding_name, "wb") as f:
        pickle.dump(embedding_docs, f)

def load_embeddings(embedding_name: str)-> list:
    """
    Charge l'embedding des documents depuis un fichier pickle.
    """
    with open(embedding_name, "rb") as f:
        return pickle.load(f)

def make_embeddings(filename: Path, embedding_function: list[Callable[[str], np.ndarray]]):
    """
    Vectorise les textes en utilisant le modèle d'embedding fourni.

    embedding_function: liste de fonctions d'encodages qui prend un texte en entrée et retourne un embedding de type <numpy.ndarray>

    Retourne un objet :
    {
        "file": "Nom du fichier",
        "path": "Chemin complet vers le fichier",
        "sections": [
            {
                "embedding": list[<numpy.ndarray>],
                "text": "Contenu du texte",
                "metadata": {
                    "title": "Titre de la section"
                }
            }
        ]
    }
    """
    embedding_doc = {
        "file": filename.name,
        "path": str(filename.resolve()),
        "sections": []
    }

    # Embeddings des documents potentiels
    if not filename.is_file() or not filename.suffix.lower() == ".md":
        raise ValueError(f"{filename} n'est pas un fichier valide.")
    
    sections = split_markdown_by_headers(filename)
    
    print(f"Vectorisation du fichier : {filename.name} ({len(sections)} sections)", flush=True)
    for section in sections:
        embedding_doc["sections"].append({
            "embedding": [
                func(section.get("content", ""))
                for func in embedding_function
            ],
            "text": section.get("content", ""),
            "metadata": {
                "title": section.get("title", ""),
            }
        })

    return embedding_doc

def md2vec(dossier: Path, embedding_function: list[Callable[[str], np.ndarray]]):
    """
    Vectorise les fichiers d'un dossier en utilisant le modèle d'embedding fourni.

    embedding_function: liste de fonctions d'encodages qui prend un texte en entrée et retourne un embedding de type <numpy.ndarray>

    Retourne un tableau d'objets: voir make_embeddings pour le format de chaque objet.
    """

    # Embeddings des documents potentiels
    for element in dossier.rglob("*"):
        if not element.is_file():
            continue

        if not element.suffix.lower() == ".md":
            continue
        
        if element.with_suffix(".vec").exists():
            print(f"Fichier : {element.name} (déjà traité)", flush=True)
            continue

        embedding_doc = make_embeddings(element, embedding_function)

        sortie = element.with_suffix(".vec")

        save_embeddings(embedding_doc, str(sortie))

md2vec(path, [embeddings_SentenceBERT, embeddings_Mistral, embeddings_FastText])


Fichier : budget_2024.md (déjà traité)
Fichier : Acceuil_affichage_Mairie de Triffouillis sur Loire.md (déjà traité)
Fichier : logo_triffouillis.md (déjà traité)
Fichier : voeux2025_trifouillis.md (déjà traité)
Fichier : Demande d'information sur l'entretien de la voirie.md (déjà traité)
Fichier : Demande de participation à la réunion municipale sur la revitalisation du centre-ville.md (déjà traité)
Fichier : Signalement et demande d'intervention pour un éclairage public défectueux.md (déjà traité)
Fichier : Festival des Rires et des Blagues.md (déjà traité)
Fichier : Fête des Saveurs Insolites.md (déjà traité)
Fichier : Journée de l’Insolence Créative.md (déjà traité)
Fichier : Le Carnaval des Objets Oubliés.md (déjà traité)
Fichier : marche_noel_trifouillis_2023.md (déjà traité)
Fichier : Nuit des Arts Urbains.md (déjà traité)
Fichier : Bulletin Municipal – Intervention Technique et Sécurité Routière.md (déjà traité)
Fichier : Bulletin Municipal – Marché de Noël 2023.md (déjà traité)

In [73]:

# Mesurer la similarité
# en utilisant la similarité cosinus sur 2 vecteurs A et B nous pouvons déterminer à quel point ils sont similaires.
# La valeur de similarité cosinus varie entre -1 et 1
# 1 signifie que les vecteurs sont identiques
# 0 signifie qu'ils sont orthogonaux (aucune similarité)
# -1 signifie qu'ils sont opposés.
#    
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def make_similarity_matrix(embedding_doc, model_index:int ):
    vectors = np.array([
        item["embedding"][model_index]
        for item in embedding_doc["sections"]
    ])

    labels = [
        f"{embedding_doc['file']} - {item['metadata']['title']}"
        for item in embedding_doc["sections"]
    ]


    similarity_matrix = cosine_similarity(vectors)

    return pd.DataFrame(
        similarity_matrix,
        index=labels,
        columns=labels
    )

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def show_vectors_distribution(embedding_doc, model_index:int, embedding_title):
    vectors = np.array([
        item["embedding"][model_index]
        for item in embedding_doc["sections"]
    ])

    corpus = [
        item['metadata']['title']
        for item in embedding_doc["sections"]
    ]

    tsne = TSNE(
        n_components=2,
        random_state=0,
        perplexity = min(5, len(vectors) - 1)
    ).fit_transform(vectors)

    plt.figure(figsize=(8, 6))

    ax = sns.scatterplot(
        x=tsne[:, 0],
        y=tsne[:, 1],
        hue=corpus,
        palette="deep"
    )


    sns.move_legend(
        ax,
        "upper left",
        bbox_to_anchor=(1, 1)
    )

    plt.title("Visualisation des embeddings avec " + embedding_title)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.show()

In [75]:

embedding_doc = load_embeddings(str(path) + "/communication/Acceuil_affichage_Mairie de Triffouillis sur Loire.vec")

similarity_df = make_similarity_matrix(embedding_doc, 0)  # 0 pour le premier modèle d'embedding (SentenceBERT)

similarity_df

,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - Chers,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - II. Horaires d'Ouverture et Contacts,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 1. Accueil et Administration,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 2. Équipes Techniques et Voirie,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Missions principales :,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 3. Service de Ramassage des Ordures,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Points forts :,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Sécurité,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Accessibilité,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Communication et Transparence,Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Participation Citoyenne,"Acceuil_affichage_Mairie de Triffouillis sur Loire.md - Fait à Triffo uillis sur Loire, Le Conseil Municipal"
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - Chers,1.000000,0.607506,0.486290,0.654445,0.454841,0.446036,0.505502,0.443206,0.476658,0.384172,0.794797,0.336514
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - II. Horaires d'Ouverture et Contacts,0.607506,1.000000,0.490909,0.438750,0.376487,0.462151,0.447082,0.378729,0.513991,0.449964,0.588997,0.509250
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 1. Accueil et Administration,0.486290,0.490909,1.000000,0.435727,0.488999,0.456261,0.443447,0.445508,0.561825,0.400186,0.472561,0.445353
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 2. Équipes Techniques et Voirie,0.654445,0.438750,0.435727,1.000000,0.471214,0.517814,0.473693,0.546606,0.408870,0.347107,0.658872,0.329581
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Missions principales :,0.454841,0.376487,0.488999,0.471214,1.000000,0.516341,0.431973,0.442580,0.501790,0.464178,0.420427,0.297524
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - 3. Service de Ramassage des Ordures,0.446036,0.462151,0.456261,0.517814,0.516341,1.000000,0.480696,0.444917,0.427715,0.363887,0.485048,0.358786
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Points forts :,0.505502,0.447082,0.443447,0.473693,0.431973,0.480696,1.000000,0.535372,0.438509,0.530265,0.386985,0.416985
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Sécurité,0.443206,0.378729,0.445508,0.546606,0.442580,0.444917,0.535372,1.000000,0.468214,0.327890,0.380490,0.448600
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Accessibilité,0.476658,0.513991,0.561825,0.408870,0.501790,0.427715,0.438509,0.468214,1.000000,0.390449,0.448559,0.349381
Acceuil_affichage_Mairie de Triffouillis sur Loire.md - · Communication et Transparence,0.384172,0.449964,0.400186,0.347107,0.464178,0.363887,0.530265,0.327890,0.390449,1.000000,0.441848,0.451425


In [78]:
len(embedding_doc["sections"][0]["embedding"])

3

In [77]:
show_vectors_distribution(embedding_doc, 0, "SentenceBERT")
show_vectors_distribution(embedding_doc, 1, "Mistral")
show_vectors_distribution(embedding_doc, 2, "FastText")


ValueError: perplexity (30.0) must be less than n_samples (12)